In [1]:
import requests
import pandas as pd
import time
from typing import Dict, List, Any

/Users/ryu/Coding/Python/RyuBell/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
def map_tmdb_genres(genre_ids: List[int]) -> str:
    genre_mapping = {
        '로맨스': [10749],
        '판타지': [14],
        '무협/사극': [36],
        '액션': [28, 12],
        '스릴러/범죄': [53, 9648, 80],
        '공포/호러': [27],
        'SF/미래': [878],
        '코미디/힐링': [35, 10751],
        '드라마/성장': [18, 16],
        '음악/스포츠': [10402]
    }

    detected = []
    for our_genre, tmdb_ids in genre_mapping.items():
        if any(gid in tmdb_ids for gid in genre_ids):
            detected.append(our_genre)

    if not detected:
        detected.append('드라마/성장')

    return ", ".join(detected[:2])

In [3]:
class TMDBPCollector:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://api.themoviedb.org/3"

    def fetch_movie_list(self, max_pages: int = 50) -> set:
        seen_ids = set()
        query_types = ['popular', 'top_rated', 'now_playing']

        print("영화 고유 ID 대량 스캔 중...")
        for q_type in query_types:
            for page in range(1, max_pages + 1):
                url = f"{self.base_url}/movie/{q_type}"
                params = {'api_key': self.api_key, 'language': 'ko-KR', 'page': page, 'region': 'KR'}
                try:
                    res = requests.get(url, params=params)
                    if res.status_code != 200: break
                    for item in res.json().get('results', []):
                        seen_ids.add(item.get('id'))
                except:
                    break
        print(f"총 {len(seen_ids)}개의 고유 영화 ID를 찾았습니다!\n")
        return seen_ids

    def fetch_premium_details(self, movie_ids: set) -> List[Dict[str, Any]]:
        rich_movies = []
        total = len(movie_ids)

        print(f"{total}개 영화의 프리미엄 데이터(키워드/배우 등) 추출 시작!")

        for idx, movie_id in enumerate(movie_ids, 1):
            url = f"{self.base_url}/movie/{movie_id}"
            params = {
                'api_key': self.api_key,
                'language': 'ko-KR',
                'append_to_response': 'keywords,credits'
            }

            try:
                res = requests.get(url, params=params)
                if res.status_code != 200: continue
                data = res.json()

                overview = data.get('overview', '').strip()
                if not overview: continue

                # 1. 장르 추출
                genres_data = data.get('genres', [])
                genre_ids = [g['id'] for g in genres_data]
                mapped_genres = map_tmdb_genres(genre_ids)

                # 2. 키워드
                keywords_data = data.get('keywords', {}).get('keywords', [])
                keywords_list = [k['name'] for k in keywords_data][:5]
                keywords_str = ", ".join(keywords_list)

                # 3. 감독 및 주연 배우
                credits = data.get('credits', {})
                cast_data = credits.get('cast', [])
                main_cast = ", ".join([c['name'] for c in cast_data][:3])

                crew_data = credits.get('crew', [])
                director = next((c['name'] for c in crew_data if c['job'] == 'Director'), 'Unknown')

                poster_path = data.get('poster_path')

                rich_movies.append({
                    'title': data.get('title'),
                    'genres': mapped_genres,
                    'overview': overview,
                    'keywords': keywords_str,            
                    'director': director,              
                    'cast': main_cast,                   
                    'runtime': data.get('runtime', 0),    
                    'popularity': data.get('popularity'), 
                    'vote_average': data.get('vote_average'),
                    'vote_count': data.get('vote_count'),
                    'release_date': data.get('release_date'),
                    'poster_url': f"https://image.tmdb.org/t/p/w500{poster_path}" if poster_path else "",
                    'movie_id': movie_id
                })

                # 진행률 표시 (100개마다 출력)
                if idx % 100 == 0:
                    print(f"  ... 진행 중: {idx} / {total} 완료")

                time.sleep(0.1)

            except Exception as e:
                print(f"Error at ID {movie_id}: {e}")
                continue

        return rich_movies

In [ ]:
if __name__ == "__main__":
    TMDB_API_KEY = "" # API 키 입력

    collector = TMDBPCollector(api_key=TMDB_API_KEY)

    # 1. 고유 ID 수집
    movie_ids = collector.fetch_movie_list(max_pages=50)

    # 2. 데이터 수집
    if movie_ids:
        raw_data = collector.fetch_premium_details(movie_ids)

        df_movies = pd.DataFrame(raw_data)
        df_movies = df_movies.sort_values(by=['popularity', 'vote_count'], ascending=[False, False]).reset_index(drop=True)

        print(f"\n 최종 정제 완료: 영화 총 {len(df_movies)}개 추출 성공!")

        csv_filename = "tmdb_movie_data.csv"
        df_movies.to_csv(csv_filename, index=False, encoding="utf-8-sig")
        print(f"파일 저장 완료: {csv_filename}")

영화 고유 ID 대량 스캔 중...
총 1820개의 고유 영화 ID를 찾았습니다!

1820개 영화의 프리미엄 데이터(키워드/배우 등) 추출 시작!
  ... 진행 중: 100 / 1820 완료
  ... 진행 중: 200 / 1820 완료
  ... 진행 중: 300 / 1820 완료
  ... 진행 중: 400 / 1820 완료
  ... 진행 중: 500 / 1820 완료
  ... 진행 중: 600 / 1820 완료
  ... 진행 중: 700 / 1820 완료
  ... 진행 중: 800 / 1820 완료
  ... 진행 중: 900 / 1820 완료
  ... 진행 중: 1000 / 1820 완료
  ... 진행 중: 1100 / 1820 완료
  ... 진행 중: 1200 / 1820 완료
  ... 진행 중: 1300 / 1820 완료
  ... 진행 중: 1500 / 1820 완료
  ... 진행 중: 1600 / 1820 완료
  ... 진행 중: 1700 / 1820 완료
  ... 진행 중: 1800 / 1820 완료

 최종 정제 완료: 영화 총 1609개 추출 성공!
파일 저장 완료: tmdb_movie_data.csv
